# NumCompute Quickstart

A short tour of each NumCompute module, with runnable examples.

**To run it:**
1. From the project root, run `uv sync` (see the README).
2. Open this notebook and choose the `.venv` Python as the kernel. If VS Code asks to install `ipykernel`, accept.
3. Run all cells from top to bottom. Paths assume the notebook runs from the `demo/` folder.

In [ ]:
from pathlib import Path

import numpy as np

DATA_DIR = Path("..") / "tests" / "test_csv_files"

## 1. Loading data (`io`)

`load_csv` reads a CSV into a 2D float array. Missing values become `NaN`.

In [ ]:
from numcompute.io import load_csv

X = load_csv(DATA_DIR / "missing_data_multiple.csv")
print(X.shape)
X

## 2. Preprocessing (`preprocessing`)

Every transformer has `fit(X)`, `transform(X)` and `fit_transform(X)`. Calling `transform` before `fit` raises `NotFittedError`.

### Filling missing values: `SimpleImputer`

In [ ]:
from numcompute.preprocessing import SimpleImputer

X_filled = SimpleImputer(fill_value=0.0).fit_transform(X)
X_filled

### Scaling: `StandardScaler` and `MinMaxScaler`

`StandardScaler` gives each column mean 0 and standard deviation 1. `MinMaxScaler` maps each column into `feature_range`.

In [ ]:
from numcompute.preprocessing import MinMaxScaler, StandardScaler

X_std = StandardScaler().fit_transform(X_filled)
print("column means:", X_std.mean(axis=0).round(6) + 0.0)
print("column stds: ", X_std.std(axis=0).round(6))

X_minmax = MinMaxScaler(feature_range=(-1, 1)).fit_transform(X_filled)
X_minmax

### Learn on training data, apply to new data

`fit` learns from one dataset; `transform` reuses what was learned on any new data with the same columns.

In [ ]:
scaler = StandardScaler().fit(X_filled)

new_data = np.array([[5.0, 5.0, 5.0]])
scaler.transform(new_data)

### Categorical data: `OneHotEncoder`

Each column becomes one 0/1 column per category (sorted). Unknown categories raise an error unless `handle_unknown="ignore"`.

In [ ]:
from numcompute.preprocessing import OneHotEncoder

colours = [["red", "S"], ["blue", "L"], ["red", "L"]]

encoder = OneHotEncoder().fit(colours)
print(encoder.categories_)
encoder.transform(colours)

In [ ]:
OneHotEncoder(handle_unknown="ignore").fit(colours).transform([["green", "S"]])

### Incremental preprocessing

Preprocessing transformers also support `partial_fit()` which allows their parameters to be updated incrementally as new chunks of data arrive.

In [ ]:
stream_scaler = StandardScaler()

chunk1 = np.array([[1.0, 10.0], [2.0, 20.0]])

chunk2 = np.array([[3.0, 30.0], [4.0, 40.0]])

stream_scaler.partial_fit(chunk1)
stream_scaler.partial_fit(chunk2)

print("Running mean: ", stream_scaler.mean_)
print(stream_scaler.transform(chunk2))

## 3. Numerical gradients (`optim`)

`grad` and `jacobian` estimate derivatives with finite differences. `method="central"` (default) is more accurate than `"forward"`.

The gradient of $f(x) = \sum x_i^2$ is $2x$:

In [ ]:
from numcompute.optim import grad, jacobian


def sum_of_squares(x):
    return np.sum(x**2)


x = np.array([1.0, 2.0, 3.0])
print("central:", grad(sum_of_squares, x))
print("forward:", grad(sum_of_squares, x, method="forward"))

For a function with several outputs, `jacobian` returns shape `(outputs, inputs)`. For $F(x) = [x_0 x_1,\ \sin x_0]$ the exact answer is $\begin{bmatrix} x_1 & x_0 \\ \cos x_0 & 0 \end{bmatrix}$:

In [ ]:
def vector_func(x):
    return np.array([x[0] * x[1], np.sin(x[0])])


jacobian(vector_func, [1.0, 2.0])

## 4. Sort and Search (`sort_search`)

`sort_search` provides stable and multi-key sorting, efficient top-k selection, manual Quickselect, and binary search.

In [ ]:
from numcompute.sort_search import (
    stable_sort,
    multi_key_sort,
    topk,
    quickselect,
    binary_search,
)

In [ ]:
### Stable sorting

values = np.array([5, 2, 4, 2, 1])

stable_sort(values)

### Multi-key sorting

`multi_key_sort` sorts the rows of a 2D array by using its column indices in order of priority. The first key is always the sorting key.

In [ ]:
data = np.array([
    [2, 30],
    [1, 20],
    [2, 10],
    [1, 10],
])

multi_key_sort(data, [0,1])

### Top-k selection

`topk` uses partial sorting to efficiently return either the largest or smallest `k` values, or their corresponding indices.

In [ ]:
values = np.array([4, 11, 2, 19, 7])

print("largest 3 values:", topk(values, 3, largest=True, return_indices=False))
print("largest 3 indices:", topk(values, 3, largest=True, return_indices=True))
print("smallest 2 values:", topk(values, 2, largest=False, return_indices=False))

### Quickselect

`quickselect` finds the k-th smallest value without fully sorting the array. `k` uses 1-based indexing. The algorithm modifies its input array in place, so copy is used below.

In [ ]:
values = np.array([6, 1, 8, 0, 5, 3])

third_smallest = quickselect(values.copy(), 3)

print("3rd smallest is:", third_smallest)

### Binary search

`binary_search` searches a sorted array and returns the index and a boolean indicating whether the value was present in the array. If the value was absent, i.e. the Boolean was False, then that index is where it is inserted.

In [ ]:
sorted_values = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9])

print("Search for 5:", binary_search(sorted_values, 5))
print("Search for 10:", binary_search(sorted_values, 10))

## 5. Pipelines (`pipeline`)

`Pipeline` chains multiple preprocessing steps such that the output of a transformer is automatically passed as the input of the next.

In [ ]:
from numcompute.pipeline import Pipeline
from numcompute.preprocessing import StandardScaler, MinMaxScaler

### Chaining preprocessing steps

`fit_transform` fits each transformer in a sequence, returning only the final transformed data.

In [ ]:
X_pipe = np.array([
    [1.0, 10.0],
    [2.0, 20.0],
    [3.0, 30.0],
])

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("minmax", MinMaxScaler()),
])

X_transformed = pipe.fit_transform(X_pipe)

X_transformed
    

### Applying fitted pipeline to new data

After fitting is completed, `transform` reuses parameters learned from original data, instead of repeating the process by fitting the preprocessing steps again.

In [ ]:
new_data = np.array([
    [1.5, 15.0],
    [2.5, 25.0],
])

pipe.transform(new_data)

A pipeline could also end up with an estimator such as `DecisionTreeClassifier` that provides `fit` and `predict`. Pipelines support `partial_fit` which allows transformers and the final estimator to be updated incrementally.

In [ ]:
from numcompute.tree import DecisionTreeClassifier

X_pipe = np.array([[1.0], [2.0], [3.0], [4.0]])

y_pipe = np.array([0, 0, 1, 1])

stream_pipe = Pipeline([
    ("scale", StandardScaler()), 
    ("model", DecisionTreeClassifier(max_depth=3, buffer_size=4)),
])

stream_pipe.partial_fit(X_pipe[:2], y_pipe[:2])
stream_pipe.partial_fit(X_pipe[2:], y_pipe[2:])

stream_pipe.predict(X_pipe)

## 6. Stats

'Stats' provides functions for basic statistical operations. Calling the appropriate function and taking in at least a numpy array is all that is required. Axis is an optional variable, with a default to 0

In [ ]:
from numcompute import mean, median, stan_dev, min, max

test_data = np.array([10, 5, 20, 15])

print(f"Data: {test_data}")
print(f"Mean: {mean(test_data)}")
print(f"Median: {median(test_data)}")
print(f"Standard Deviation: {stan_dev(test_data)}")
print(f"Minimum: {min(test_data)}")
print(f"Maximum: {max(test_data)}")

Histogram and quantile have bin and probability as options they also have defaults however these defaults are too large for this test data so they are included here

In [ ]:
from numcompute import histogram, quantile

test_data = np.array([10, 5, 20, 15])
test_bins = 4
test_prob = 0.5

print(f"Data: {test_data} Bins: {test_bins} Probability:{test_prob}")
print(f"Histogram: {histogram(test_data,test_bins)}")
print(f"Quantile: {quantile(test_data,test_prob)}")

## Streaming Statistics

`StreamingStats` updates the statistics incrementally as new chunks of data arrive, allowing the statistics to be calculated incrementally instead of having to process the entire dataset in a single cell. 

In [44]:
from numcompute import StreamingStats

stat = StreamingStats()

stat.update_stats(np.array([[1.0, 10.0], [2.0, 20.0]]))
stat.update_stats(np.array([[3.0, 30.0], [4.0, 40.0]]))

print("Mean: ", stat.mean())
print("Variance: ", stat.variance())
print("Minimum: ", stat.min())
print("Maximum: ", stat.max())

Mean:  [ 2.5 25. ]
Variance:  [  1.25 125.  ]
Minimum:  [ 1. 10.]
Maximum:  [ 4. 40.]


## 7. Metrics

The NumCompute package provides useful metrics that can be updated incrementally. By calling the `update` processes a new chunk of predictions while `result` and returns the metric accumulatated across all chunks.

In [ ]:
from numcompute.metrics import Accuracy, Precision, Recall, F1, ConfusionMatrix, MSE

pred_array = np.array([1,1,0,0])
true_array = np.array([1,0,0,0])

metrics = [
    Accuracy(),
    Precision(),
    Recall(),
    F1(),
    ConfusionMatrix(),
]
for metric in metrics:
    metric.update(true_array, pred_array)
    print(type(metric).__name__,":", metric.result())

Accuracy : 0.75
Precision : 0.5
Recall : 1.0
F1 : 0.6666666666666666
ConfusionMatrix : (np.int64(1), np.int64(1), np.int64(2), np.int64(0))


### Metrics across chunks

Metric state is preserved between calls to `update`

In [ ]:
accuracy = Accuracy()

accuracy.update(np.array([1, 0]), np.array([1, 1]))
accuracy.update(np.array([1, 0]), np.array([1,0]))

print("Cumulative accuracy: ", accuracy.result())

In [ ]:
mse = MSE()
mse.update(np.array([1,0.5,1,0]), np.array([2,1,0.5,0]))

print(f"Mean Squared Error: {mse.result()}")

In [ ]:
from numcompute.metrics import AUC, RollingWindowMetric

auc = AUC()

auc.update(np.array([0, 0]), np.array([0.1, 0.2]))
auc.update(np.array([1, 1]), np.array([0.8, 0.9]))

print(f"AUC: {auc.result()}")

roll_acc = RollingWindowMetric(Accuracy(), window_size=3)

roll_acc.update(np.array([1, 0, 1, 0]), np.array([1, 1, 0, 0]))

print(f"Rolling Accuracy: {roll_acc.result()}")

## 8. Rank
The ranking tools in this library, compute average, ordinal and dense rankings for data as well as an additional percentile function to be able to identify values at given positions/percentiles. If a position lies between two points, interpolation methods such as linear, lower, higher and midpoint interpolations are used to find the percentile.


In [ ]:
from numcompute import rank, percentile

test_array = np.array([0, 2, 3, 2, 7, 2])

# Predicted Ordinal ranking
pred_arr_ord = np.array([1, 2, 5, 3, 6, 4])
# Predicted Average Ranking
pred_arr_avg = np.array([1, 3, 5, 3, 6, 3])
# Predicted Dense Ranking
pred_arr_dense = np.array([1, 2, 3, 2, 4, 2,])
# Predicted linear 30th Percentile (q= 0.3)
pred_percentile = 2.0

print(f"Predicted values:\nAverage ranks: {pred_arr_avg}\nOrdinal ranks: {pred_arr_ord}\nDense ranks: {pred_arr_dense}")
print(f"\nPredicted Percentile, q = 30 (or 0.3), linear interpolation: {pred_percentile}")

print(f"\nComputed values:\nAverage ranks: {rank(test_array, "average")}\nOrdinal ranks: {rank(test_array, "ordinal")}\nDense ranks: {rank(test_array, "dense")}")
print(f"\nComputed percentile, q = 30 (or 0.3), linear interpolation: {percentile(test_array,0.3, "linear")}")


## 9. Decision Tree

`DecisionTreeClassifier` implements a depth-limited classification tree that uses either Gini impurity or entropy. 

For streaming data, `partial_fit` stores incoming observations in a buffer, so when `buffer_size` observations have been accumulated, the tree is retrained using the buffered data.

In [ ]:
from numcompute.tree import DecisionTreeClassifier

X_tree = np.array([[1.0], [2.0], [3.0], [4.0], [5.0], [6.0]])

y_tree = np.array([0, 0, 0, 1, 1, 1])

tree = DecisionTreeClassifier(max_depth=3, criterion="gini")

tree.fit(X_tree, y_tree)

print(f"Predictions: {tree.predict(X_tree)}")

In [ ]:
stream_tree = DecisionTreeClassifier(max_depth=3, buffer_size=6)

stream_tree.partial_fit(X_tree[:3], y_tree[:3])
stream_tree.partial_fit(X_tree[3:], y_tree[3:])

print(f"Streaming Prediction: {stream_tree.predict(X_tree)}")

## 10. Streaming Training

`StreamTrainer` manages incremental model training and evaluation, where it records per-chunk accuracy, cumulative accuracy and an approximate memory footprint.

In [ ]:
from numcompute.stream import  StreamTrainer

model = DecisionTreeClassifier(max_depth=3, buffer_size=3)

trainer = StreamTrainer(model)

X_chunk1 = X_tree[:3]
y_chunk1 = y_tree[:3]

X_chunk2 = X_tree[3:]
y_chunk2 = y_tree[3:]

trainer.fit_chunk(X_chunk1, y_chunk1)

print(trainer.score_chunk(X_chunk1, y_chunk1))

trainer.fit_chunk(X_chunk2, y_chunk2)

print(trainer.score_chunk(X_chunk2, y_chunk2))

trainer.get_logs()

## 11. Visualisation

The NumCompute package also provide reusable Matplotlib function that show streaming metrics, comparing models, and comparing predictions with ground truth. 

In [ ]:
from numcompute.visualise import (
    compare_models, plot_metric_over_time, plot_predictions_vs_ground_truth
)

metric_values = [0.55, 0.65, 0.72, 0.80]

plot_metric_over_time(metric_values, "Accuracy over time", "Accuracy")


In [ ]:
compare_models([0.50, 0.62, 0.71, 0.78], [0.55, 0.68, 0.75, 0.84], ["Model 1", "Model 2"])

In [ ]:
plot_predictions_vs_ground_truth(np.array([0, 0, 1, 1]), np.array([0, 1, 1, 1]))

## 12. Benchmarking
The numcompute library also includes a Benchmarking tool called "benchit" which measures the average execution time of any function that is plugged into it. It also has a separate demo mode that enables compares various numcompute functions, which are implemented via vectorization, to a Python loop equivalent.


For very small inputs, execution times can be noisy and overhead can dominate
the result, so these timings should not be interpreted as representative of
large-scale performance.

In [ ]:
from numcompute import benchmarking
DEMO_FUNCS = ("stable_sort", "multi_key_sort", "median", "stan_dev", "imputer")

for func in DEMO_FUNCS:
    benchmarking.benchit(func,mode='demo', repeat=10)

### Streaming benchmark

The streaming benchmark compares the loop-based and vectorised incremental statistics used by `StandardScaler`.

In [ ]:
from numcompute.benchmarking import(
    benchmark_streaming_scaler,
    benchmark_buffer_sizes
)

rng = np.random.default_rng(42)

X_benchmark = rng.normal(size=(10_000, 10))

scaler_results = benchmark_streaming_scaler(X_benchmark, size_of_chunk=500, repeat=5)

scaler_results

### Decision Tree buffer-size benchmark

Different buffer size are evaluated to see how frequently the streaming decision tree is retrained, the benchmark records the execution time and the final accuracy.

In [ ]:
X_tree_benchmark = rng.normal(size=(1000, 4))

y_tree_benchmark = (X_tree_benchmark[:, 0] + X_tree_benchmark[:, 1] > 0).astype(int)

buffer_results = benchmark_buffer_sizes(
    X_tree_benchmark,
    y_tree_benchmark, 
    buffer_sizes=[50, 100, 250],
    size_of_chunk=50,
    repeat=3,
    max_depth=5
)

buffer_results

Benchmark environment: Python 3.14.4. Results are machine-specific and may vary depending on hardware and system load.